# 기말 프로젝트 트랙 3 시작 노트북. Guidance 크기
*Final project, Track 3 starter. The guidance scale*

**확인할 가설 / Hypothesis to check.** guidance 크기 w 가 커지면 표본이 주문한 클래스와 더 자주 맞고, 덜 퍼진다.
*As the guidance scale w grows, samples match their class more often and spread less.*

12주차 실습과 같은 망 · 학습 · 표집입니다(겹치는 세 클래스, 클래스 2 를 주문). 기준은 실습 그대로(클래스 중심 거리 1.5, 3,000회 학습, 조건 누락 확률 0.15)이고, 여러분은 **조건 하나**(클래스 거리 또는 학습 길이)를 바꿔 w 를 다시 돌립니다. 학습은 시드 셋(29 · 30 · 31)으로 자동 반복합니다. **가설이 맞지 않아도 근거를 갖춰 설명하면 같은 점수입니다.**
*The Week 12 lab's network, training and sampling (three overlapping classes, asking for class 2). The baseline is the lab setting (class-center distance 1.5, 3,000 updates, drop probability 0.15); you change **one condition** (class distance or training length) and sweep w again. Training repeats over three seeds automatically. **A hypothesis that does not hold, explained with evidence, earns the same marks.***

**쓰는 법 / How to use.** 「내 선택」 칸에서 조건과 값을 고르고 작성칸 1 을 쓴 뒤 런타임 > 모두 실행 → 끝의 작성칸 2 ~ 4 와 AI 사용 칸에 답. 고르기 전에는 기준 설정만 돕니다. 전체 1~2분(CPU). 조건 누락 확률까지 바꾸는 것은 두 요인 실험이라 그림 아래 선택 칸으로 따로 두었습니다. 필수가 아닙니다.
*Choose a condition and value in "Your choice", write answer 1, then Runtime > Run all, and fill answer cells 2 to 4 and AI use at the end. Before you choose, only the baseline runs. One to two minutes on a CPU. Changing the drop probability as well is a two-factor experiment, so it sits in an optional cell after the figures. It is not required.*

In [ ]:
#@title 준비 / Setup { display-mode: "form" }
import math, time, json, hashlib, platform, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt
NOTEBOOK = "FP-Track3 v7"

torch.set_num_threads(2)                      # 실습과 같음. 같은 컴퓨터에서 다시 돌리면 같은 숫자 / as in the labs
GREEN, GOLD, RED, GRAY = "#406551", "#C9971C", "#B04A3A", "#9AA5A0"
T = 100
betas = torch.cat([torch.zeros(1), torch.linspace(0.0008, 0.08, T)])
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

def t_features(t):
    tt = (t.float()[:, None] - 1) / T
    freqs = (2 ** torch.arange(8).float())[None, :]
    return torch.cat([torch.sin(tt * freqs * math.pi), torch.cos(tt * freqs * math.pi)], 1)

def jump(x0, eps, abar_t):
    return torch.sqrt(abar_t) * x0 + torch.sqrt(1 - abar_t) * eps

def ddpm_mean(x_t, eps_hat, beta_t, alpha_t, abar_t):
    return (x_t - beta_t / torch.sqrt(1 - abar_t) * eps_hat) / torch.sqrt(alpha_t)

K, NULL, TARGET = 3, 3, 2

class CondNet(nn.Module):
    """12주차 실습의 조건부 잡음 예측기. 번호 3 = null / the Week 12 conditional predictor; index 3 = null."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 + 16 + K + 1, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 2))
    def forward(self, x, t, y):
        return self.net(torch.cat([x, t_features(t), nn.functional.one_hot(y, K + 1).float()], 1))

def make_data(distance, gen):
    """12주차 실습의 세 클래스. 중심 거리 distance, 흩어짐 0.8, 3,000점 / the lab's three classes at the given center distance."""
    a = torch.tensor([90., 210., 330.]) * math.pi / 180
    C = torch.stack([distance * torch.cos(a), distance * torch.sin(a)], 1)
    y = torch.randint(0, K, (3000,), generator=gen)
    return C[y] + torch.randn(3000, 2, generator=gen) * 0.8, y, C

def train(data, labels, updates, p_drop, seed):
    """12주차 실습 §2 의 학습: 가끔 라벨을 null 로 / the lab's training: sometimes the label becomes null."""
    gen = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    model = CondNet(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(updates):
        idx = torch.randint(0, len(data), (512,), generator=gen); x0, y = data[idx], labels[idx]
        t = torch.randint(1, T + 1, (len(x0),), generator=gen); eps = torch.randn(x0.shape, generator=gen)
        drop = torch.rand(len(x0), generator=gen) < p_drop
        y_in = torch.where(drop, torch.full_like(y, NULL), y)
        loss = (model(jump(x0, eps, abar[t][:, None]), t, y_in) - eps).square().mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def sample_cfg(model, n, y, w, gen):
    """12주차 실습 §4: 걸음마다 두 번 예측해 ε_u + w(ε_c − ε_u) 로 합친다 / the lab's CFG sampling."""
    x = torch.randn(n, 2, generator=gen)
    yy, yn = torch.full((n,), y), torch.full((n,), NULL)
    for t in range(T, 0, -1):
        tt = torch.full((n,), t)
        eps_c, eps_u = model(x, tt, yy), model(x, tt, yn)
        x = ddpm_mean(x, eps_u + w * (eps_c - eps_u), betas[t], alphas[t], abar[t])
        if t > 1: x = x + torch.sqrt(betas[t]) * torch.randn(n, 2, generator=gen)
    return x

def measure(pts, C):
    """실습의 세 지표: 클래스 2 비율(가장 가까운 중심, %), 퍼짐(표준편차, 두 좌표 평균), 밀림(표본 평균과 진짜 중심 2 의 거리).
    The lab's three measures: class-2 share (nearest center, %), spread (std, mean of the two coordinates), drift (sample mean to the true center 2)."""
    return (100 * (torch.cdist(pts, C).argmin(1) == TARGET).float().mean().item(), pts.std(0).mean().item(), (pts.mean(0) - C[TARGET]).norm().item())
print("준비 완료 / Setup ready | torch", torch.__version__)

## 내 선택 / Your choice
조건 하나만 바꿉니다. 기준은 「거리 1.5, 3,000회, 누락 확률 0.15」 입니다. *Change one condition. The baseline is distance 1.5, 3,000 updates, drop probability 0.15.*

| 조건 / condition | 기준 / baseline | 바꿀 수 있는 값 / new value |
|---|---|---|
| class distance | 1.5 (Week 12 lab) | 1.0 (more overlap) or 2.5 (less overlap) |
| training length | 3,000 updates | 1,000 or 6,000 updates |

w = 0 · 1 · 3 · 6 · 12, 표본 600개(시드 101), 학습 시드 29 · 30 · 31 은 고정입니다. 시드 29 의 기준 결과는 12주차 실습의 숫자와 같습니다(요약 셀의 CHECK 줄).
*w = 0, 1, 3, 6, 12, the 600 samples (seed 101) and training seeds 29, 30, 31 are fixed. Seed 29 of the baseline reproduces the Week 12 lab (the CHECK line in the summary cell).*

| 지표 / measure | 뜻 / meaning (12주차 실습과 같음 / as in the Week 12 lab) |
|---|---|
| `class2_%` | 클래스 2 비율. 가장 가까운 중심이 클래스 2 인 표본의 비율<br>Class-2 share: the share of samples whose nearest center is class 2 |
| `spread` | 퍼짐. 표본 전체의 두 좌표 표준편차의 평균<br>Spread: the mean of the two coordinates' standard deviations over all samples |
| `drift` | 밀림. 표본 평균과 진짜 클래스 2 중심 사이의 거리<br>Drift: the distance from the sample mean to the true class-2 center |

In [ ]:
#@title 내 선택 / Your choice { display-mode: "form" }
CONDITION = "choose one"  #@param ["choose one", "class distance", "training length"]
NEW_CLASS_DISTANCE = 1.0  #@param [1.0, 2.5] {type:"raw"}
NEW_TRAINING_UPDATES = 6000  #@param [1000, 6000] {type:"raw"}

BASELINE = {"distance": 1.5, "updates": 3000, "p_drop": 0.15}
CHANGED = dict(BASELINE)
if CONDITION == "class distance": CHANGED["distance"] = float(NEW_CLASS_DISTANCE)
if CONDITION == "training length": CHANGED["updates"] = int(NEW_TRAINING_UPDATES)
SETTINGS = [("baseline", BASELINE)] + ([("changed", CHANGED)] if CONDITION != "choose one" else [])
CHOICE = {"condition": CONDITION, "baseline": BASELINE, "changed": CHANGED if CONDITION != "choose one" else None}
for name, s in SETTINGS: print(f"{name:9s}: distance {s['distance']} | {s['updates']} updates | drop {s['p_drop']}")
if CONDITION == "choose one":
    print("아직 조건을 고르지 않았습니다. 기준 설정만 돌립니다 / No condition chosen yet: only the baseline runs.")

## 작성칸 1. 비교 목적과 판정 기준 / Answer 1. Purpose and decision rule
「내 선택」 을 정한 뒤, 그 설정으로 돌리기 전에 아래 칸에 쓰세요. 제안서의 평가 계획을 옮겨 와도 됩니다. 결과를 본 뒤에 고치지 말고, 결과를 본 뒤에 썼다면 그렇다고 밝히세요.
*Write this in the cell below after choosing and before you run your choice; you may carry over the evaluation plan from your proposal. Do not revise it after seeing results, and if you wrote it after seeing results, say so.*

- **목적 / Purpose.** 바꾼 조건과 값, 고정한 것, 이 비교로 답하려는 질문. 제안서와 달라진 설정이 있으면 무엇이 왜 달라졌는지.
  *The condition and values you change, what stays fixed, and the question this comparison answers; if any setting differs from your proposal, what changed and why.*
- **예측 / Prediction.** 지표마다 어느 쪽으로 움직일지와 그렇게 보는 이유.
  *For each measure, which way it will move and why you think so.*
- **판정 기준 / Decision rule.** 「더 자주 맞고 덜 퍼진다」 를 어느 지표로 판정할지, 지표끼리 어긋나면 무엇을 먼저 볼지.
  *Which measures decide "match their class more often and spread less", and which comes first if they disagree.*

**답 / Answer.** 여기에 쓰세요. *Write here.*

In [ ]:
#@title 실행 / Run. 1~2분 / one to two minutes { display-mode: "form" }
WS, SEEDS = (0, 1, 3, 6, 12), (29, 30, 31)

def run(settings):
    rows, ex = [], {}
    for label, s in settings:
        data, labels, C = make_data(s["distance"], torch.Generator().manual_seed(29))
        real2 = data[labels == TARGET]
        sh, sp, dr = measure(real2, C)
        rows.append({"setting": label, "seed": "real class 2", "w": -1, "class2_%": sh, "spread": sp, "drift": dr})
        ex[(label, "real")] = (real2, C)
        for seed in SEEDS:
            model = train(data, labels, s["updates"], s["p_drop"], seed)
            for w in WS:
                pts = sample_cfg(model, 600, TARGET, float(w), torch.Generator().manual_seed(101))
                sh, sp, dr = measure(pts, C)
                rows.append({"setting": label, "seed": seed, "w": w, "class2_%": sh, "spread": sp, "drift": dr})
                if seed == SEEDS[0]: ex[(label, w)] = (pts, C)
    return pd.DataFrame(rows), ex

t0 = time.time()
RESULTS, examples = run(SETTINGS)
RESULTS.to_csv("track3_results.csv", index=False)
print(f"완료 / done in {time.time() - t0:.0f} s. 결과는 track3_results.csv 에도 저장 / also saved to track3_results.csv")

In [ ]:
#@title 요약 / Summary { display-mode: "form" }
def summarize(R):
    g = R[R["seed"] != "real class 2"]
    cols = ["class2_%", "spread", "drift"]
    s = pd.concat([g.groupby(["setting", "w"])[c].agg(lambda v: f"{v.mean():.2f} ({v.min():.2f}–{v.max():.2f})") for c in cols], axis=1)
    s.columns = [c + " mean (min–max over 3 seeds)" for c in cols]
    print("SUMMARY. w 별 평균과 시드 범위 / mean and seed range per w"); print(s.to_string()); print()
    print("REAL CLASS 2. 같은 지표를 진짜 클래스 2 에서 / the same measures on the real class 2")
    print(R[R["seed"] == "real class 2"][["setting"] + cols].round(3).to_string(index=False))
summarize(RESULTS)
b29 = RESULTS[(RESULTS["setting"] == "baseline") & (RESULTS["seed"] == 29)].set_index("w")[["class2_%", "spread", "drift"]]
print("\nCHECK. 기준 설정 · 시드 29 = 12주차 실습 / baseline, seed 29 = the Week 12 lab"); print(b29.T.round(2).to_string())
# 기대값 / Expected   (12주차 실습 §5 의 w 표와 같은 숫자 / the same numbers as the w table in section 5 of the Week 12 lab)
#   w             0      1       3      6      12
#   class2_%  33.00  91.67  100.00 100.00 100.00   (실습은 정수로 33 · 92 · 100 · 100 · 100 % / the lab prints whole numbers)
#   spread     1.28   0.82    0.48   0.32    0.20
#   drift      1.41   0.10    0.90   1.03    1.01
#   진짜 클래스 2 는 91 % · 0.81 · 0.02 (REAL CLASS 2 표) / the real class 2: 91 %, 0.81, 0.02 (REAL CLASS 2 table)

In [ ]:
#@title 그림 / Figures { display-mode: "form" }
for label, _ in SETTINGS:
    fig, axs = plt.subplots(1, len(WS), figsize=(2.4 * len(WS), 2.7))
    real2, C = examples[(label, "real")]
    for ax, w in zip(axs, WS):
        pts, _ = examples[(label, w)]
        ax.scatter(real2[:, 0], real2[:, 1], s=2, color=GRAY, alpha=0.3)
        ax.scatter(pts[:, 0], pts[:, 1], s=3, color=GREEN, alpha=0.5)
        ax.scatter(C[:, 0], C[:, 1], marker="+", s=60, color="black", linewidths=1.3)
        ax.set(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5), xticks=[], yticks=[], title=f"w = {w}"); ax.set_aspect("equal")
    plt.suptitle(f"{label}: 600 samples asking for class 2, seed 29 (gray = real class 2, + = centers)", fontsize=10); plt.tight_layout(); plt.show()
g = RESULTS[RESULTS["seed"] != "real class 2"]
fig, axs = plt.subplots(1, 3, figsize=(12.6, 3.0))
for ax, c in zip(axs, ["class2_%", "spread", "drift"]):
    for label, col in zip([s[0] for s in SETTINGS], (GREEN, RED)):
        part = g[g["setting"] == label]
        for seed in SEEDS:
            p = part[part["seed"] == seed]; ax.plot(p["w"], p[c], color=col, alpha=0.35, lw=1)
        m = part.groupby("w")[c].mean(); ax.plot(m.index, m.values, color=col, lw=2.2, marker="o", label=label)
        ax.axhline(RESULTS[(RESULTS["setting"] == label) & (RESULTS["seed"] == "real class 2")][c].iloc[0], color=col, ls=":", lw=1)
    ax.set(xlabel="guidance scale w", title=c); ax.legend(fontsize=8, frameon=False)
plt.suptitle("Thin lines: each training seed. Thick: mean. Dotted: real class 2.", fontsize=10); plt.tight_layout(); plt.show()

## 선택 확장. 조건 누락 확률 / Optional extension. The drop probability
필수가 아닙니다. 위 비교를 누락 확률만 바꿔 한 번 더 돌리면 「조건 × 누락 확률」 의 두 요인 실험이 됩니다. 썼다면 작성칸 2 · 3 에 두 요인이라고 밝히세요. 1~2분 더 걸립니다.
*Not required. Rerunning the comparison above with another drop probability makes a two-factor experiment (condition × drop probability). If you use it, say so in answers 2 and 3. It adds one to two minutes.*

In [ ]:
#@title 선택 확장 / Optional extension { display-mode: "form" }
RUN_EXTENSION = False  #@param {type:"boolean"}
EXT_P_DROP = 0.3  #@param [0.05, 0.3] {type:"raw"}
if RUN_EXTENSION:
    ext_settings = [(f"{label}, drop {EXT_P_DROP}", dict(s, p_drop=float(EXT_P_DROP))) for label, s in SETTINGS]
    EXT_RESULTS, _ = run(ext_settings)
    EXT_RESULTS.to_csv("track3_extension_results.csv", index=False)
    summarize(EXT_RESULTS)
else:
    print("선택 확장을 돌리지 않았습니다 / The optional extension was not run.")

In [ ]:
#@title 기록 / Record. 출력을 지우지 마세요 / keep this output { display-mode: "form" }
# 결과 해시는 같은 결과인지 대조하는 용도입니다. 실행 시간은 빼고 셉니다 / the results hash is for comparing results; run times are left out
record = {"notebook": NOTEBOOK, "choice": CHOICE,
          "results_sha256": hashlib.sha256(RESULTS.drop(columns=["seconds"], errors="ignore").to_csv(index=False).encode()).hexdigest()[:16],
          "python": platform.python_version(), "torch": torch.__version__, "device": "CPU (GPU available: " + ("yes" if torch.cuda.is_available() else "no") + ")",
          "run_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}
if "RUN_EXTENSION" in globals():                                   # 트랙 3 의 선택 확장 / Track 3's optional extension
    record["extension"] = ({"p_drop": float(EXT_P_DROP), "results_sha256": hashlib.sha256(EXT_RESULTS.to_csv(index=False).encode()).hexdigest()[:16]}
                           if RUN_EXTENSION and "EXT_RESULTS" in globals() else "not run")
print(json.dumps(record, ensure_ascii=False, indent=2))

## 이 노트북이 보고서입니다 / This notebook is your report
아래 작성칸 2 ~ 4 와 AI 사용 칸에 답을 쓰고, 출력을 저장한 채 내세요. 숫자는 위 표에서 그대로 인용합니다. 위 기록 셀의 출력은 지우지 마세요(재현성 1점).
*Answer in cells 2 to 4 and the AI-use cell below, and submit with the outputs saved. Quote numbers from the tables above. Keep the output of the record cell above (reproducibility, 1 point).*

포스터에는 요약 표와 그림 하나를 옮기면 됩니다. 표와 그림을 옮긴 것만으로는 분석 점수를 받지 않습니다.
*For the poster, copy the summary table and one figure. Copying tables and figures alone does not earn the analysis marks.*

## 작성칸 2. 내 결과와 판정 / Answer 2. Your results and judgement
- **관찰 / Observation.** 표와 그림에서 본 것을 숫자로. 기준 설정과 바꾼 설정을 나란히.
  *What the tables and figures show, in numbers, with the baseline next to your changed setting.*
- **가능한 설명 / Possible explanation.** 왜 그렇게 나왔다고 보는지. 관찰과 구분하고, 확인하지 않은 설명이면 그렇다고 씁니다.
  *Why you think it came out this way, kept apart from the observation; say so if you have not tested the explanation.*
- **판정 / Judgement.** 작성칸 1 의 판정 기준으로 가설이 맞았는지. 가장 중요한 결론을, 그것을 받치는 숫자 둘과 그 숫자가 나온 설정으로 씁니다(필수 문단).
  *Did the hypothesis hold under your decision rule in answer 1? Give your main conclusion with the two numbers that support it and the setting they come from (the required paragraph).*
- 진짜 클래스 2 의 값(REAL CLASS 2 표, 그림의 점선)과 견주어 읽으세요.
  *Read the measures against the real class 2 (the REAL CLASS 2 table and the dotted lines in the figures).*

**답 / Answer.** 여기에 쓰세요. *Write here.*

## 작성칸 3. 한계와 판단을 바꿀 증거 / Answer 3. Limits, and evidence that would change your judgement
- 이 실행이 확인한 것과 확인하지 못한 것. 예를 들어 표집 시드만 바꾼 것은 학습이 안정적인지를 확인하지 않습니다.
  *What these runs checked and what they did not; for example, changing only the sampling seed does not test whether training is stable.*
- 결론이 맞지 않을 수 있는 상황.
  *Situations where the conclusion may not hold.*
- 어떤 증거가 나오면 판단을 바꾸겠는지.
  *What evidence would make you change your judgement.*
- 노트북을 바꿨다면 무엇을 바꿨는지(예: 시드 추가).
  *Anything you changed in the notebook, such as adding seeds.*

**답 / Answer.** 여기에 쓰세요. *Write here.*

## 작성칸 4. 받은 질문과 답 / Answer 4. The question you received, and your answer
12/16 까지 포스터 댓글로 받은 질문 하나를 옮겨 적고, 이 노트북의 결과를 근거로 답하세요.
*Copy one question you received in the comments on your poster by 16 Dec, and answer it with evidence from this notebook.*

**답 / Answer.** 여기에 쓰세요. *Write here.*

## AI 사용 / AI use
도구, 용도, 대략의 범위를 쓰세요. 쓰지 않았으면 「없음」. 결론과 그 근거는 본인의 판단이어야 합니다.
*The tool, its purpose and the rough extent; write "none" if you used none. The conclusions and the evidence for them must be your own judgement.*

**답 / Answer.** 여기에 쓰세요. *Write here.*